## Problem Statement
- To predict the acceptance of H-1B visa step 1

## Import the required packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import xgboost as xgb
from xgboost.sklearn import XGBClassifier
from sklearn import metrics   #Additional scklearn functions
from sklearn.model_selection import GridSearchCV

## Get the Data

In [ ]:
df = pd.read_csv("h1b_2011_2016.csv")

In [ ]:
df

## Data Cleansing and Exploratory Data Analysis

In [ ]:
# get the columns
df.columns

In [ ]:
# check the unique case status
df['YEAR'].unique()

In [ ]:
# get the first five rows
df.head()

In [ ]:
# get the last five rows
df.tail()

In [ ]:
# delete the unnamed, lon and lat columns 
df = df.drop(['Unnamed: 0'], axis=1)

In [ ]:
# get the general information
df.info()

In [ ]:
# get the na values of each column
df.isna().sum()

In [ ]:
# delete all the rows having na values
df.dropna(axis=0, inplace=True)
df.isna().sum()

In [ ]:
# get the count of values according to year
df['YEAR'].value_counts()

In [ ]:
# change the data type of year to object
df['YEAR'] = df['YEAR'].astype(int)
#df.info()

In [ ]:
# get the statistical information
df.describe()

In [ ]:
# split the worksite column into city and state
# delete the worksite column
df[['CITY', 'STATE']] = df['WORKSITE'].str.split(', ', 1, expand=True)
del df['WORKSITE']

In [ ]:
# get the unique case status
df['CASE_STATUS'].unique()

In [ ]:
# get the count of each unique case status
df['CASE_STATUS'].value_counts()

In [ ]:
#from sklearn.preprocessing import LabelEncoder

#status_encoder = LabelEncoder()
#df['CASE_STATUS'] = status_encoder.fit_transform(df['CASE_STATUS'])
#df.head(15)

In [ ]:
# convert Certified case status into 1 and Certified-Withdrawn, Denied, Withdrawn, Rejected, Invalidated
# and Pending Quality and Compliance Review - Unassigned to 0
from sklearn.preprocessing import label_binarize

df['ACCEPT_REJECT'] = label_binarize(df['CASE_STATUS'], classes = ['CERTIFIED'])

In [ ]:
df.rename(columns={'lon': 'LON', 'lat': 'LAT'}, inplace=True)
df.columns

In [ ]:
# change the position of the accept_reject, city and state columns
df = df[['CASE_STATUS', 'EMPLOYER_NAME', 'SOC_NAME', 'JOB_TITLE', 'FULL_TIME_POSITION', 'PREVAILING_WAGE', 'YEAR', 'STATE', 'CITY', 'LON', 'LAT', 'ACCEPT_REJECT']]
df.head()

In [ ]:
# delete the case status column
del df['CASE_STATUS']

In [ ]:
# unique employer names
df['EMPLOYER_NAME'].value_counts().count()

In [ ]:
# unique soc names
df['SOC_NAME'].value_counts().count()

In [ ]:
# unique job titles
df['JOB_TITLE'].value_counts().count()

In [ ]:
# remove punctuation marks from the employer_name column 
df['EMPLOYER_NAME'] = df['EMPLOYER_NAME'].str.replace("""[,./'")(*]""", "")
df['EMPLOYER_NAME'] = df['EMPLOYER_NAME'].str.replace("-", " ")

In [ ]:
# remove punctuation marks from the soc_name column 
df['SOC_NAME'] = df['SOC_NAME'].str.replace("""[,./'")(*]""", "")
df['SOC_NAME'] = df['SOC_NAME'].str.replace("-", " ")

In [ ]:
# remove punctuation marks from the job_title column 
df['JOB_TITLE'] = df['JOB_TITLE'].str.replace("""[,./'")(*]""", "")
df['JOB_TITLE'] = df['JOB_TITLE'].str.replace("-", " ")

In [ ]:
# change the contents of all string columns to upper case

df['EMPLOYER_NAME'] = df['EMPLOYER_NAME'].str.upper()
df['SOC_NAME'] = df['SOC_NAME'].str.upper()
df['JOB_TITLE'] = df['JOB_TITLE'].str.upper()
df['FULL_TIME_POSITION'] = df['FULL_TIME_POSITION'].str.upper()
df['CITY'] = df['CITY'].str.upper()
df['STATE'] = df['STATE'].str.upper()

In [ ]:
# unique employer names
df['EMPLOYER_NAME'].value_counts().count()

In [ ]:
# unique soc names
df['SOC_NAME'].value_counts().count()

In [ ]:
# unique job titles
df['JOB_TITLE'].value_counts().count()

In [ ]:
#emp_name = df.sort_values(by='EMPLOYER_NAME')
#emp_name.head(10)

In [ ]:
from sklearn.preprocessing import LabelEncoder

status_encoder = LabelEncoder()
df['FULL_TIME_POSITION'] = status_encoder.fit_transform(df['FULL_TIME_POSITION'])
#df.head(15)

In [ ]:
# display the first n values of dataset
df.head()

In [ ]:
df['ACCEPT_REJECT'].value_counts()

In [ ]:
df['SOC_NAME'].value_counts().count()

In [ ]:
df['SOC_NAME'].value_counts()

In [ ]:
soc_values = df['SOC_NAME'].value_counts()

In [ ]:
soc_values.describe()

In [ ]:
soc_values[soc_values>500].count()

In [ ]:
#counts1 = df['SOC_NAME'].value_counts()
df = df[~df['SOC_NAME'].isin(soc_values[soc_values < 501].index)]

In [ ]:
df['SOC_NAME'].value_counts().count()

In [ ]:
df['JOB_TITLE'].value_counts().count()

In [ ]:
df['JOB_TITLE'].value_counts()

In [ ]:
job_values = df['JOB_TITLE'].value_counts()

In [ ]:
job_values.describe()

In [ ]:
job_values[job_values>1000].count()

In [ ]:
#counts2 = df['JOB_TITLE'].value_counts()
df = df[~df['JOB_TITLE'].isin(job_values[job_values < 1001].index)]

In [ ]:
df['JOB_TITLE'].value_counts().count()

In [ ]:
df['EMPLOYER_NAME'].value_counts().count()

In [ ]:
df['EMPLOYER_NAME'].value_counts()

In [ ]:
emp_values = df['EMPLOYER_NAME'].value_counts()

In [ ]:
emp_values.describe()

In [ ]:
emp_values[emp_values>600].count()

In [ ]:
#counts3 = df['EMPLOYER_NAME'].value_counts()
df = df[~df['EMPLOYER_NAME'].isin(emp_values[emp_values < 601].index)]

In [ ]:
df['EMPLOYER_NAME'].value_counts().count()

In [ ]:
df['CITY'].value_counts().count()

In [ ]:
df['CITY'].value_counts()

In [ ]:
city_values = df['CITY'].value_counts()

In [ ]:
city_values.describe()

In [ ]:
city_values[city_values>600].count()

In [ ]:
#counts4 = df['CITY'].value_counts()
df = df[~df['CITY'].isin(city_values[city_values < 601].index)]

In [ ]:
df['CITY'].value_counts().count()

In [ ]:
soc_values = df['SOC_NAME'].value_counts()
soc_values[soc_values>500].count()

In [ ]:
for x in soc_values[soc_values<500].index:
    df.loc[df['SOC_NAME']==x, 'SOC_NAME'] = 'OTHERS'

In [ ]:
df['SOC_NAME'].value_counts().head(10)

In [ ]:
df['SOC_NAME'].value_counts().count()

In [ ]:
job_values = df['JOB_TITLE'].value_counts()
job_values[job_values>500].count()

In [ ]:
for x in job_values[job_values<500].index:
    df.loc[df['JOB_TITLE']==x, 'JOB_TITLE'] = 'OTHERS'

In [ ]:
df['JOB_TITLE'].value_counts().head(10)

In [ ]:
df['JOB_TITLE'].value_counts().count()

In [ ]:
emp_values = df['EMPLOYER_NAME'].value_counts()
emp_values[emp_values>600].count()

In [ ]:
for x in emp_values[emp_values<600].index:
    df.loc[df['EMPLOYER_NAME']==x, 'EMPLOYER_NAME'] = 'OTHERS'

In [ ]:
df['EMPLOYER_NAME'].value_counts().head(10)

In [ ]:
df['EMPLOYER_NAME'].value_counts().count()

In [ ]:
city_values = df['CITY'].value_counts()
city_values[city_values>700].count()

In [ ]:
for x in city_values[city_values<700].index:
    df.loc[df['CITY']==x, 'CITY'] = 'OTHERS'

In [ ]:
df['CITY'].value_counts().head(10)

In [ ]:
df['CITY'].value_counts().count()

In [ ]:
state_values = df['STATE'].value_counts()
state_values[state_values>10].count()

In [ ]:
for x in state_values[state_values<10].index:
    df.loc[df['STATE']==x, 'STATE'] = 'OTHERS'

In [ ]:
df['STATE'].value_counts().count()

In [ ]:
df['ACCEPT_REJECT'].value_counts()

In [ ]:
df.tail(20)

In [ ]:
#df.to_csv("step1_tab.csv")

In [ ]:
df = pd.read_csv("step1_tab.csv")

In [ ]:
#df = pd.read_csv("step1_new.csv")

In [ ]:
del df['Unnamed: 0']

In [ ]:
df

In [ ]:
#employer_values = df['EMPLOYER_NAME'].value_counts()
#for x in employer_values[employer_values.isin(range(10,101))].index:
#    df.loc[df['EMPLOYER_NAME']==x, 'EMPLOYER_NAME'] = 'OTHERS'

In [ ]:
df['EMPLOYER_NAME'] = df['EMPLOYER_NAME'].astype("category")
df['SOC_NAME'] = df['SOC_NAME'].astype("category")
df['JOB_TITLE'] = df['JOB_TITLE'].astype("category")
df['STATE'] = df['STATE'].astype("category")
df['CITY'] = df['CITY'].astype("category")

In [ ]:
employer_dict = dict(enumerate(df['EMPLOYER_NAME'].cat.categories))
soc_dict = dict(enumerate(df['SOC_NAME'].cat.categories))
jobtitle_dict = dict(enumerate(df['JOB_TITLE'].cat.categories))
state_dict = dict(enumerate(df['STATE'].cat.categories))
city_dict = dict(enumerate(df['CITY'].cat.categories))

In [ ]:
#employer_dict.items()

In [ ]:
list(employer_dict.values()).index('INTUIT INC')

In [ ]:
soc_dict.get(5)

In [ ]:
list(soc_dict.values()).index('WEB DEVELOPERS')

In [ ]:
jobtitle_dict.get(122)

In [ ]:
list(jobtitle_dict.values()).index('TECHNOLOGY ANALYST   US')

In [ ]:
state_dict.get(20)

In [ ]:
list(state_dict.values()).index('NEW YORK')

In [ ]:
city_dict.get(43)

In [ ]:
list(city_dict.values()).index('MOUNTAIN VIEW')

In [ ]:
df['PREVAILING_WAGE'].sort_values().tail(30)

In [ ]:
df['EMPLOYER_NAME'] = df['EMPLOYER_NAME'].cat.codes
df['SOC_NAME'] = df['SOC_NAME'].cat.codes
df['JOB_TITLE'] = df['JOB_TITLE'].cat.codes
df['STATE'] = df['STATE'].cat.codes
df['CITY'] = df['CITY'].cat.codes

In [ ]:
df.info()

In [ ]:
# check the covariance
df.cov()

In [ ]:
# check the correlation
df.corr()

In [ ]:
# Import library for VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calc_vif(X):

    # Calculating VIF
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return(vif)

In [ ]:
X = df.iloc[:,:-1]
calc_vif(X)

In [ ]:
X = df.drop(['YEAR', 'FULL_TIME_POSITION', 'LON', 'LAT', 'ACCEPT_REJECT'],axis=1)
calc_vif(X)

In [ ]:
x = df.drop(['LON', 'LAT', 'ACCEPT_REJECT'], axis=1)

y = df['ACCEPT_REJECT']

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_regression

select = SelectKBest(score_func=chi2, k=6)
z = select.fit_transform(x,y)

filter = select.get_support()
features = x.columns
 
print("All features:")
print(features)
 
print("Selected best 6:")
print(features[filter])

In [ ]:
#features = df.iloc[:,[0,1,2,4,6,7]].values
#features

In [ ]:
#labels = df.iloc[:,-1].values
#labels

In [ ]:
#features = df[['EMPLOYER_NAME', 'SOC_NAME', 'JOB_TITLE', 'PREVAILING_WAGE', 'STATE', 'CITY']].values

In [ ]:
#labels = df['ACCEPT_REJECT'].values

In [ ]:
x = df.drop(['FULL_TIME_POSITION', 'YEAR', 'LON', 'LAT', 'ACCEPT_REJECT'], axis=1)

y = df['ACCEPT_REJECT']
#y = df.drop(['EMPLOYER_NAME', 'SOC_NAME', 'JOB_TITLE', 'PREVAILING_WAGE', 'STATE', 'CITY', 'FULL_TIME_POSITION', 'YEAR', 'LON', 'LAT'], axis=1)

In [ ]:
y

In [ ]:
y_train.value_counts()

In [ ]:
#y['ACCEPT_REJECT'].value_counts()

In [ ]:
#y_train_sm

In [ ]:
#x = features
#y = labels

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)

In [ ]:
print("Before Over Sampling, count of the label '1': {}".format(sum(y_train == 1)))
print("Before Over Sampling, count of the label '0': {} \n".format(sum(y_train == 0)))

from imblearn.over_sampling import SMOTE

sm = SMOTE()
x_train_sm, y_train_sm = sm.fit_resample(x_train, y_train.ravel())

print('After Over Sampling, the shape of the train_X: {}'.format(x_train_sm.shape))
print('After Over Sampling, the shape of the train_y: {} \n'.format(y_train_sm.shape))
print("After Over Sampling, count of the label '1': {}".format(sum(y_train_sm == 1)))
print("After Over Sampling, count of the label '0': {}".format(sum(y_train_sm == 0)))

In [ ]:
#from sklearn.preprocessing import StandardScaler

#scaler = StandardScaler()
#x_train_ss = scaler.fit_transform(x_train_sm)
#x_test_ss = scaler.transform(x_test)

In [ ]:
#from sklearn.decomposition import PCA

# reduce the 3 independent variables to only two
#pca = PCA(n_components=2)
#x_test_pca = pca.fit_transform(x_test_ss)

In [ ]:
#x_test_pca

In [ ]:
#def viz_model_dt():
#    from sklearn.tree import DecisionTreeClassifier
    
#    model = DecisionTreeClassifier()
#    model.fit(x_test_pca, y_test)
#    return model

In [ ]:
#def viz_model_rf():
#    from sklearn.ensemble import RandomForestClassifier
    
#    model = RandomForestClassifier(n_estimators=10)
#    model.fit(x_test_pca, y_test)
#    return model

In [ ]:
#def viz_model_xgb():
#    import xgboost as xgb
    
#    model = xgb.XGBClassifier()
#    model.fit(x_test_pca, y_test)
#    return model

In [ ]:
#viz_dt = viz_model_dt()

In [ ]:
#viz_rf = viz_model_rf()

In [ ]:
#viz_xgb = viz_model_xgb()

In [ ]:
#plt.scatter(x_test_pca[:,0][y_test == 0], x_test_pca[:,1][y_test == 0], color="red", label="Not Approved")
#plt.scatter(x_test_pca[:,0][y_test == 1], x_test_pca[:,1][y_test == 1], color="green", label="Approved")

#plt.title("Classification Result Visualization")
#plt.legend()

In [ ]:
#y_pred = viz_dt.predict(x_test_pca)

#plt.scatter(x_test_pca[:,0][y_pred == 0], x_test_pca[:,1][y_pred == 0], color="red", label="Not Approved")
#plt.scatter(x_test_pca[:,0][y_pred == 1], x_test_pca[:,1][y_pred == 1], color="green", label="Approved")

#plt.title("Classification Result Visualization Pred")
#plt.legend()

In [ ]:
#def create_model_lg():
#    from sklearn.linear_model import LogisticRegressionCV
    
#    model = LogisticRegressionCV(max_iter=1000)
#    model.fit(x_train_sm, y_train_sm)
#    return model

In [ ]:
#def create_model_nb():
#    from sklearn.naive_bayes import GaussianNB
    
#    model = GaussianNB()
#    model.fit(x_train_sm, y_train_sm)
#    return model

In [ ]:
def create_model_knn():
    from sklearn.neighbors import KNeighborsClassifier
    
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(x_train_sm, y_train_sm)
    return model

In [ ]:
def create_model_dt():
    from sklearn.tree import DecisionTreeClassifier
    
    model = DecisionTreeClassifier()
    model.fit(x_train_sm, y_train-sm)
    return model

In [ ]:
def create_model_rf():
    from sklearn.ensemble import RandomForestClassifier
    
    model = RandomForestClassifier()
    model.fit(x_train_sm, y_train_sm)
    return model

In [ ]:
def create_model_xgb():
    import xgboost as xgb
    
    model = xgb.XGBClassifier()
    model.fit(x_train_sm, y_train_sm)
    return model

In [ ]:
def modelfit(alg, useTrainCV=True, cv_folds=5):
    
    if useTrainCV:
        xgb_param = alg.get_xgb_params()
        xgtrain = xgb.DMatrix(x_train_sm.values, label=y_train_sm)
        cvresult = xgb.cv(xgb_param, xgtrain, num_boost_round=alg.get_params()['n_estimators'], nfold=cv_folds,
                          metrics='auc', verbose_eval=False)
        alg.set_params(n_estimators=cvresult.shape[0])
    
    #Fit the algorithm on the data
    alg.fit(x_train_sm, y_train_sm)
        
    #Predict training set:
    dtrain_predictions = alg.predict(x_test)
    dtrain_predprob = alg.predict_proba(x_test)[:,1]
        
    #Print model report:
    print("\nModel Report")
    print("Accuracy : %.4g" % metrics.accuracy_score(y_test, dtrain_predictions))
    print("AUC Score (Train): %f" % metrics.roc_auc_score(y_test, dtrain_predprob))
                    
    feat_imp = pd.Series(alg.get_booster().get_fscore()).sort_values(ascending=False)
    feat_imp.plot(kind='bar', title='Feature Importances')
    plt.ylabel('Feature Importance Score')
    
    print("Best Iteration: {}".format(alg.get_booster().best_iteration))
    print("Best N Tree Limit: {}".format(alg.get_booster().best_ntree_limit))

In [ ]:
#no_columns = 'FULL_TIME_POSITION', 'YEAR', 'LON', 'LAT'
#target = 'ACCEPT_REJECT'

#Choose all predictors except target & no_columns
#predictors = [x for x in x_train_sm.columns if x not in [target, no_columns]]
xgb1 = xgb.XGBClassifier(
    learning_rate =0.3,
    n_estimators=1000,
    max_depth=5,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8, 
    objective= 'binary:logistic',
    nthread=4,
    scale_pos_weight=1,
    seed=27,
    eval_metric='auc'
)

modelfit(xgb1)

In [ ]:
#model_lg = create_model_lg()

In [ ]:
#model_nb = create_model_nb()

In [ ]:
model_knn = create_model_knn()

In [ ]:
model_dt = create_model_dt()

In [ ]:
model_rf = create_model_rf()

In [ ]:
model_xgb = create_model_xgb()

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score, classification_report

def evaluate_model(model, model_name):
    print(f"--- evaluation report of {model_name} ---")
    
    y_pred = model.predict(x_test)
    
    # get confusion matrix
    print("Confusion Matrix")
    print(confusion_matrix(y_test, y_pred))
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"accuaracy = {accuracy * 100:0.2f}%")
    
    precision = precision_score(y_test, y_pred)
    print(f"precision = {precision}") 
    
    recall = recall_score(y_test, y_pred)
    print(f"recall = {recall}")
    
    f1 = f1_score(y_test, y_pred)
    print(f"f1 acore = {f1}")
    
    # get the classification report
    print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

def roc_evaluation(model, model_name):
    print(f"--- roc evaluation report of {model_name} ---")
    
    # get the probabilities
    y_pred_probabilities = model.predict_proba(x_test)[:, 1]
    
    # get fpr, tpr
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_probabilities)
    
    # draw the curv
    plt.plot(fpr, tpr, label="RoC Curve")
    plt.legend()
    
    # get the roc score
    roc_score = roc_auc_score(y_test, y_pred_probabilities)
    print(f"roc score = {roc_score}")

In [ ]:
#evaluate_model(model_lg, "Logistic Regression")
#roc_evaluation(model_lg, "Logistic Regression")

In [ ]:
#evaluate_model(model_nb, "Naive Bayes")
#roc_evaluation(model_nb, "Naive Bayes")

In [ ]:
evaluate_model(model_knn, "K Nearest Neighbour")
roc_evaluation(model_knn, "K Nearest Neighbour")

In [ ]:
evaluate_model(model_dt, "Decision Tree")
roc_evaluation(model_dt, "Decision Tree")

In [ ]:
evaluate_model(model_rf, "Random Forest")
roc_evaluation(model_rf, "Random Forest")

In [ ]:
evaluate_model(model_xgb, "XGBoost")
roc_evaluation(model_xgb, "XGBoost")

In [ ]:
#import tensorflow as tf
#from keras.models import Sequential
#from keras.layers import Dense

#model = Sequential()
#model.add(Dense(6, activation='relu', input_shape=(6,)))
#model.add(Dense(6, activation='relu'))
#model.add(Dense(1, activation='sigmoid'))
#model.compile(loss='binary_crossentropy',
#optimizer='adam',
#metrics=['accuracy'])
#model.fit(x_train_ss, y_train_sm, epochs=50, batch_size=10, verbose=1)

In [ ]:
#EMPLOYER_NAME, SOC_NAME, JOB_TITLE, PREVAILING_WAGE, STATE, CITY
#predictions_dt = model_dt.predict([[81, 24, 76, 100000, 10, 66]])
#probabilities_dt = model_dt.predict_proba([[81, 24, 76, 100000, 10, 66]])
#predictions

#print("---- Decision Tree ---- ")
#print(predictions_dt)
#print(probabilities_dt)
#if predictions_dt[0] >= 0.5:
#    print(f"The visa is approved with {probabilities_dt}")
#else:
#    print("The visa is not approved")  
#65, 23, 32, 90000, 16, 45
#3, 47, 22, 70000, 3, 34

In [ ]:
#EMPLOYER_NAME, SOC_NAME, JOB_TITLE, PREVAILING_WAGE, STATE, CITY
predictions_rf = model_rf.predict([[65, 48, 122, 120000, 20, 43]])
probabilities_rf = model_rf.predict_proba([[65, 48, 122, 120000, 20, 43]])
#predictions

print("---- Random Forest ---- ")
print(predictions_rf)
print(probabilities_rf)
if predictions_rf[0] >= 0.5:
    print(f"The visa is approved with {round(probabilities_rf[0][1] * 100, 2)}% probability")
else:
    print(f"The visa is not approved with {round(probabilities_rf[0][0] * 100, 2)}% probabiltiy")  

In [ ]:
#EMPLOYER_NAME, SOC_NAME, JOB_TITLE, PREVAILING_WAGE, STATE, CITY
predictions_xgb = model_xgb.predict(np.array([[65, 48, 122, 200000, 20, 43]]))
probabilities_xgb = model_xgb.predict_proba(np.array([[65, 48, 122, 200000, 20, 43]]))

print("---- XGBoost ---- ")
print(predictions_xgb)
print(probabilities_xgb)

if predictions_xgb[0] >= 0.5:
    print(f"The visa is approved with {round(probabilities_xgb[0][1] * 100, 2)}% probability")
else:
    print(f"The visa is not approved with {round(probabilities_xgb[0][0] * 100, 2)}% probability")
    
# 34, 5, 122, 60000, 43, 43
# 65, 23, 32, 90000, 16, 45

In [ ]:
import pickle

# create an empty binary file 
model_file = open('h1b_prediction_model_rf2.pk', 'wb')

# dump the model into the file
pickle.dump(model_rf, model_file)

# close the file
model_file.close()